# TP3 — 03. Análisis temporal, solar y antecedentes de error

**Taller de Análisis de Datos I · Notebook revisado para `TP3/notebooks/`**

**Preguntas:** ¿cómo varía el error original según hora UTC, hora solar, fecha y estación? ¿Qué errores históricos podrían conocerse en una emisión posterior? ¿Qué conclusiones respaldan incorporar variables temporales al modelado?

**Unidad:** pronóstico de temperatura a 12 o 24 h con `match_type="interval"` y temperaturas finitas. Error firmado = **observado − pronosticado**; sesgo positivo = el pronóstico subestima la temperatura. MAE = media del valor absoluto.

**Alcance estricto:** inventario 61 CSV; **48 históricos** (10/09–01/11), cinco días excluidos (22–26/10 por pérdida de información y recomendación docente), ocho días **reservados** (02–09/11), nunca usados en este EDA. Las fechas del archivo y la fecha objetivo se verifican, no se presumen equivalentes.

**Integración:** utiliza el **Parquet histórico del Notebook 00** cuando existe (`datos_modelado/consolidados/tp3_historico_interval_12_24.parquet`); alternativamente lee una sola vez los 48 CSV por bloques. No necesita ejecutar Notebook 01 o 02 en la misma sesión. Exporta datos para el dashboard a `resultados/dashboard/03_temporal_solar/` sin modificar modelos, datos brutos, resultados de 04–08 ni el test.

**Advertencia:** los lags de objetivo fijo 12/24 h que se estudian aquí son **exploratorios**. El Notebook 04 usa un algoritmo diferente (`lag_error_disponible_c`, edad e indicador de ausencia), basado en el último error disponible bajo un supuesto de latencia; **no se deben confundir sus coberturas ni sustituirlos en el modelo final**.

In [ ]:
# 03.1 — Dependencias, raíz y manifiesto temporal compartido
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import gc
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

try:
    from IPython.display import display
except ImportError:
    display = print

RAIZ_PROYECTO = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "dataset").is_dir() and (p / "tp3_config.py").is_file()), None
)
if RAIZ_PROYECTO is None:
    raise FileNotFoundError("No se encuentra TP3/ con dataset/ y tp3_config.py.")
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from tp3_config import obtener_particiones, HISTORICO, EXCLUIDOS, TEST

particiones_03 = obtener_particiones(RAIZ_PROYECTO, exigir_61=True)
archivos_03 = particiones_03["historico"]
archivos_test_03 = particiones_03["test"]
assert (len(particiones_03["inventario"]), len(archivos_03),
        len(particiones_03["excluidos"]), len(archivos_test_03)) == (61, 48, 5, 8)
assert set(archivos_03).isdisjoint(archivos_test_03)
assert len(HISTORICO & TEST) == 0

DIR_03 = RAIZ_PROYECTO / "resultados" / "dashboard" / "03_temporal_solar"
PARQUET_00 = (RAIZ_PROYECTO / "datos_modelado" / "consolidados"
              / "tp3_historico_interval_12_24.parquet")
KPI_00 = (RAIZ_PROYECTO / "resultados" / "dashboard"
          / "00_calidad_datos" / "13_auditoria_kpis_resumen.json")
RUN_ID_03 = datetime.now(timezone.utc).strftime("tp3_03_%Y%m%dT%H%M%SZ")

print("Raíz:", RAIZ_PROYECTO)
print("Inventario: 61 | Histórico: 48 | Excluidos: 5 | Test reservado: 8")
print("Parquet histórico 00:", "disponible" if PARQUET_00.exists() else "ausente: se usarán CSV históricos")
print("El test NO se abrirá en este notebook.")

## 1. Construcción única de la cohorte histórica

Se emplean **exactamente las mismas reglas que el Notebook 00**: horizontes 12/24, emparejamiento `interval` y temperaturas pronosticada/observada finitas. La prioridad es reutilizar el consolidado existente para evitar una segunda lectura de los CSV. Si falta, se utiliza una única pasada por bloques y se procesan juntas las variables del análisis solar y de lags. Ningún procedimiento lee datos de noviembre.

Las marcas temporales se convierten a UTC. Se exige concordancia entre la fecha objetivo y el día histórico declarado para impedir un cambio inadvertido de cohorte. La auditoría de calidad detallada sigue perteneciendo al Notebook 00.

In [ ]:
# 03.2 — Carga única: Parquet de 00 o una sola pasada de los CSV históricos
COLUMNAS_03 = [
    "station_id", "latitude", "longitude", "snapshot_time_utc",
    "collection_time_utc", "forecast_generated_at_utc", "target_time_utc",
    "observation_time_utc", "observation_collection_hour_utc",
    "horizon_target_hours", "match_type", "temperature_forecast_c",
    "temperature_observed_c", "archivo_origen"
]
FECHAS_03 = [
    "snapshot_time_utc", "collection_time_utc", "forecast_generated_at_utc",
    "target_time_utc", "observation_time_utc", "observation_collection_hour_utc"
]
CLAVES_00_03 = ["station_id", "snapshot_time_utc", "target_time_utc", "horizon_target_hours"]
ESPERADO_HIST_03 = 854009
ESPERADO_HORIZ_03 = {12: 427009, 24: 427000}


def cargar_historico_filtrado() -> tuple[pd.DataFrame, str]:
    """Lee una sola fuente histórica. Nunca lee el Parquet/CSV reservado de test."""
    motor_parquet = (importlib.util.find_spec("pyarrow") is not None
                    or importlib.util.find_spec("fastparquet") is not None)
    if PARQUET_00.is_file() and motor_parquet:
        df = pd.read_parquet(PARQUET_00, columns=COLUMNAS_03)
        fuente = "Parquet histórico exportado por 00"
    else:
        if PARQUET_00.is_file() and not motor_parquet:
            print("Existe Parquet 00 pero no hay motor Parquet; se leen los CSV históricos por bloques.")
        piezas = []
        columnas_csv = [c for c in COLUMNAS_03 if c != "archivo_origen"]
        for numero, archivo in enumerate(archivos_03, 1):
            for bloque in pd.read_csv(
                archivo, usecols=columnas_csv, chunksize=100_000, low_memory=False
            ):
                t_pron = pd.to_numeric(bloque["temperature_forecast_c"], errors="coerce")
                t_obs = pd.to_numeric(bloque["temperature_observed_c"], errors="coerce")
                mascara = (
                    bloque["horizon_target_hours"].isin([12, 24])
                    & bloque["match_type"].eq("interval")
                    & np.isfinite(t_pron) & np.isfinite(t_obs)
                )
                seleccion = bloque.loc[mascara].copy()
                if len(seleccion):
                    seleccion["archivo_origen"] = archivo.name
                    piezas.append(seleccion)
            if numero % 12 == 0:
                print(f"Lectura histórica: {numero}/48 archivos")
        if not piezas:
            raise ValueError("No se hallaron filas históricas válidas")
        df = pd.concat(piezas, ignore_index=True)
        del piezas
        gc.collect()
        fuente = "48 CSV históricos; una pasada por bloques"

    for columna in FECHAS_03:
        df[columna] = pd.to_datetime(df[columna], utc=True, errors="coerce")
    for columna in ["temperature_forecast_c", "temperature_observed_c", "latitude", "longitude"]:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")
    df["horizon_target_hours"] = pd.to_numeric(df["horizon_target_hours"], errors="coerce")
    df["station_id"] = df["station_id"].astype("string")
    # Control defensivo también cuando el Parquet existe.
    df = df.loc[
        df["horizon_target_hours"].isin([12, 24])
        & df["match_type"].eq("interval")
        & np.isfinite(df["temperature_forecast_c"])
        & np.isfinite(df["temperature_observed_c"])
    ].copy()
    df.sort_values(["target_time_utc", "station_id", "horizon_target_hours"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df, fuente


df_temporal, fuente_03 = cargar_historico_filtrado()
assert len(df_temporal) == ESPERADO_HIST_03, (
    f"Cohorte histórica distinta: {len(df_temporal)} vs. {ESPERADO_HIST_03}. "
    "No continuar hasta reconciliar con el Notebook 00."
)
assert df_temporal["horizon_target_hours"].value_counts().to_dict() == ESPERADO_HORIZ_03
assert df_temporal[CLAVES_00_03].notna().all().all()
assert not df_temporal.duplicated(CLAVES_00_03).any()
assert df_temporal["target_time_utc"].notna().all()
assert set(df_temporal["target_time_utc"].dt.date).issubset(HISTORICO)
assert df_temporal["archivo_origen"].isin([p.name for p in archivos_03]).all()

fecha_de_archivo = pd.to_datetime(
    df_temporal["archivo_origen"].str.extract(r"(\d{4}_\d{2}_\d{2})", expand=False)
    .str.replace("_", "-", regex=False), errors="coerce", utc=True
)
assert fecha_de_archivo.notna().all()
assert df_temporal["target_time_utc"].dt.normalize().eq(fecha_de_archivo).all(), (
    "La fecha objetivo no corresponde al archivo: auditar antes de continuar."
)

# Reconciliación opcional con la huella exportada por Notebook 00.
HUELLA_03 = hashlib.sha256(
    pd.util.hash_pandas_object(
        df_temporal[CLAVES_00_03], index=False
    ).to_numpy(dtype="uint64").tobytes()
).hexdigest()
if KPI_00.is_file():
    kpi00 = json.loads(KPI_00.read_text(encoding="utf-8"))
    assert HUELLA_03 == kpi00["historico_claves_sha256"], (
        "Las claves históricas no coinciden con la exportación 00."
    )

print("FUENTE:", fuente_03)
print("Filas:", len(df_temporal))
print("Horizontes:")
display(df_temporal["horizon_target_hours"].value_counts().sort_index())
print("Días históricos:", df_temporal["target_time_utc"].dt.date.nunique())
print("Huella de claves (primeros 16):", HUELLA_03[:16])

## 2. Consistencia de marcas temporales y hora objetivo

`target_time_utc` expresa la **hora de inicio del intervalo objetivo** cuando coincide con `snapshot_time_utc + horizon_target_hours`. La observación puede ocurrir más tarde dentro de ese intervalo. No debe sustituirse silenciosamente por la hora de observación, porque eso cambiaría la pregunta de pronóstico.

Se mide explícitamente la proporción de objetivos con minutos/segundos no nulos; no se presupone que todos están exactamente en `HH:00`. La **hora solar media aproximada** será `UTC decimal + longitud/15`, módulo 24. No corresponde a hora civil ni incluye ecuación del tiempo.

El instante operacional de inferencia podría ser posterior al `snapshot` nominal: las marcas de captura y generación se comparan para mostrarlo. Esas marcas no prueban cuándo un dato estuvo disponible de verdad en el sistema meteorológico.

In [ ]:
# 03.3 — Auditoría de reloj y de disponibilidad (todo el histórico)
df_temporal["error_c"] = (df_temporal["temperature_observed_c"] -
                           df_temporal["temperature_forecast_c"])
df_temporal["error_abs_c"] = df_temporal["error_c"].abs()
assert np.isfinite(df_temporal[["error_c", "error_abs_c"]].to_numpy()).all()

df_temporal["horizonte_real_h"] = (
    df_temporal["target_time_utc"] - df_temporal["snapshot_time_utc"]
).dt.total_seconds() / 3600

df_temporal["obs_desde_objetivo_min"] = (
    df_temporal["observation_time_utc"] - df_temporal["target_time_utc"]
).dt.total_seconds() / 60

df_temporal["generacion_desde_snapshot_min"] = (
    df_temporal["forecast_generated_at_utc"] - df_temporal["snapshot_time_utc"]
).dt.total_seconds() / 60

df_temporal["captura_desde_snapshot_min"] = (
    df_temporal["collection_time_utc"] - df_temporal["snapshot_time_utc"]
).dt.total_seconds() / 60

df_temporal["recoleccion_desde_observacion_min"] = (
    df_temporal["observation_collection_hour_utc"] - df_temporal["observation_time_utc"]
).dt.total_seconds() / 60

df_temporal["emision_efectiva_utc"] = df_temporal[
    ["collection_time_utc", "forecast_generated_at_utc"]
].max(axis=1)
# Si falta una de las dos marcas, no queda acreditada una emisión efectiva completa.
ambas_marcas_emision = df_temporal[
    ["collection_time_utc", "forecast_generated_at_utc"]
].notna().all(axis=1)
df_temporal.loc[~ambas_marcas_emision, "emision_efectiva_utc"] = pd.NaT

hora_objetivo_no_entera = (
    df_temporal["target_time_utc"].dt.minute.ne(0)
    | df_temporal["target_time_utc"].dt.second.ne(0)
    | df_temporal["target_time_utc"].dt.microsecond.ne(0)
)


def auditoria_relojes(grupo: pd.DataFrame) -> pd.Series:
    """Métricas temporales por horizonte, sin inferir latencia real de ingestión."""
    return pd.Series({
        "registros": len(grupo),
        "objetivos_fuera_hora_entera": int(hora_objetivo_no_entera.loc[grupo.index].sum()),
        "horizonte_no_coincide": int((~np.isclose(
            grupo["horizonte_real_h"], grupo["horizon_target_hours"], atol=1e-7
        )).sum()),
        "observaciones_fuera_0_a_60_min": int(
            (~grupo["obs_desde_objetivo_min"].between(0, 60, inclusive="left")).sum()
        ),
        "generacion_posterior_snapshot": int((grupo["generacion_desde_snapshot_min"] > 0).sum()),
        "captura_posterior_snapshot": int((grupo["captura_desde_snapshot_min"] > 0).sum()),
        "emision_efectiva_faltante": int(grupo["emision_efectiva_utc"].isna().sum()),
        "mediana_obs_desde_objetivo_min": grupo["obs_desde_objetivo_min"].median(),
        "mediana_generacion_desde_snapshot_min": grupo["generacion_desde_snapshot_min"].median(),
        "mediana_captura_desde_snapshot_min": grupo["captura_desde_snapshot_min"].median(),
        "max_recoleccion_desde_observacion_min": grupo["recoleccion_desde_observacion_min"].max(),
    })

reloj_filas = []
for horizonte, grupo in df_temporal.groupby("horizon_target_hours"):
    fila = auditoria_relojes(grupo).to_dict()
    fila["horizonte_h"] = int(horizonte)
    reloj_filas.append(fila)
df_auditoria_relojes_03 = pd.DataFrame(reloj_filas).sort_values("horizonte_h")
print("AUDITORÍA TEMPORAL HISTÓRICA:")
display(df_auditoria_relojes_03.round(3))
print("Objetivos en HH:00:00:", int((~hora_objetivo_no_entera).sum()),
      "/", len(df_temporal))
print("Nota: los objetivos no enteros, si existen, se conservan y se usan con su precisión real.")

## 3. Hora UTC, hora solar y codificación cíclica

La hora del objetivo se expresa en décimas/segundos de UTC cuando corresponda y se transforma por la longitud geográfica, sin usar el momento de observación. Seno y coseno permiten continuidad entre 23:59 y 00:00. La muestra histórica comprende una ventana de aproximadamente dos meses: las variables de día del año del Notebook 04 son una **codificación de calendario**, no una estacionalidad anual demostrada en este notebook.

In [ ]:
# 03.4 — Variables cíclicas y controles geométricos

def calcular_hora_solar(hora_utc_decimal: pd.Series, longitud: pd.Series) -> pd.Series:
    """Hora solar media aproximada desde longitud (grados este) y UTC decimal."""
    return ((hora_utc_decimal + longitud / 15) % 24).where(longitud.between(-180, 180))

fecha_obj = df_temporal["target_time_utc"]
df_temporal["hora_utc_decimal"] = (
    fecha_obj.dt.hour + fecha_obj.dt.minute / 60
    + fecha_obj.dt.second / 3600 + fecha_obj.dt.microsecond / 3_600_000_000
)
df_temporal["hora_utc"] = fecha_obj.dt.hour
df_temporal["hora_solar_decimal"] = calcular_hora_solar(
    df_temporal["hora_utc_decimal"], df_temporal["longitude"]
)
df_temporal["hora_solar"] = np.floor(df_temporal["hora_solar_decimal"]).astype("Int64")
df_temporal["fecha_utc"] = fecha_obj.dt.normalize()

for prefijo, columna in [
    ("hora_utc", "hora_utc_decimal"), ("hora_solar", "hora_solar_decimal")
]:
    angulo = 2 * np.pi * df_temporal[columna] / 24
    df_temporal[f"{prefijo}_sin"] = np.sin(angulo)
    df_temporal[f"{prefijo}_cos"] = np.cos(angulo)
    identidad = df_temporal[f"{prefijo}_sin"]**2 + df_temporal[f"{prefijo}_cos"]**2
    assert np.allclose(identidad.dropna().to_numpy(dtype=float), 1, atol=1e-12)

print("Longitudes válidas:", int(df_temporal["longitude"].between(-180, 180).sum()))
print("Horas UTC presentes:", sorted(df_temporal["hora_utc"].dropna().unique().tolist()))
print("Horas solares presentes:", sorted(df_temporal["hora_solar"].dropna().unique().tolist()))
print("Ejemplos de codificación cíclica:")
display(df_temporal[[
    "station_id", "horizon_target_hours", "hora_utc_decimal",
    "longitude", "hora_solar_decimal", "hora_utc_sin", "hora_utc_cos",
    "hora_solar_sin", "hora_solar_cos"
]].sample(n=min(8, len(df_temporal)), random_state=42).round(3))

## 4. Comportamiento horario y sesgo de composición

Calcularemos sesgo, MAE, número de estaciones y fechas por **hora y horizonte**. Un pico de error con pocas estaciones no representa necesariamente el comportamiento geográfico completo. Se dejarán visibles los horarios sin observaciones (no se interpolan valores faltantes).

In [ ]:
# 03.5 — Tablas de hora UTC y solar, conservando denominadores

def resumir_por_hora(datos: pd.DataFrame, nombre_hora: str) -> pd.DataFrame:
    """Resume error, cobertura de estaciones y fechas en una escala horaria."""
    return (
        datos.dropna(subset=[nombre_hora])
        .groupby(["horizon_target_hours", nombre_hora], observed=True)
        .agg(registros=("error_c", "size"), estaciones=("station_id", "nunique"),
             dias=("fecha_utc", "nunique"), sesgo_medio_c=("error_c", "mean"),
             mae_c=("error_abs_c", "mean"))
        .reset_index().rename(columns={"horizon_target_hours": "horizonte_h"})
        .sort_values(["horizonte_h", nombre_hora])
    )

resumen_utc_03 = resumir_por_hora(df_temporal, "hora_utc")
resumen_solar_03 = resumir_por_hora(df_temporal, "hora_solar")
assert resumen_utc_03["registros"].sum() == len(df_temporal)
assert resumen_solar_03["registros"].sum() == int(df_temporal["hora_solar"].notna().sum())
print("POR HORA UTC (muestra):")
display(resumen_utc_03.head(8).round(3))
print("POR HORA SOLAR (muestra):")
display(resumen_solar_03.head(8).round(3))
print("Horas UTC ausentes por horizonte:")
for h in [12, 24]:
    presentes = set(resumen_utc_03.loc[resumen_utc_03.horizonte_h.eq(h), "hora_utc"].astype(int))
    print(f"  {h} h:", sorted(set(range(24)) - presentes))
print("Menor cobertura solar por horizonte:")
display(resumen_solar_03.sort_values(["horizonte_h", "estaciones"]).groupby("horizonte_h").head(4).round(3))

In [ ]:
# 03.6 — Gráficos horarios comparables (sin valores inventados)
def graficar_horas(tabla: pd.DataFrame, columna: str, titulo: str):
    """Devuelve figura con sesgo, MAE y cobertura de estaciones por hora."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for h in [12, 24]:
        g = tabla.loc[tabla.horizonte_h.eq(h)].set_index(columna).reindex(range(24))
        for ax, col in zip(axes, ["sesgo_medio_c", "mae_c", "estaciones"]):
            ax.plot(g.index, g[col], marker="o", markersize=3, label=f"{h} h")
    axes[0].axhline(0, linestyle="--", color="gray", linewidth=1)
    for ax, name, unidad in zip(
        axes, ["Sesgo medio", "MAE", "Estaciones representadas"], ["°C", "°C", "cantidad"]
    ):
        ax.set_title(name)
        ax.set_ylabel(unidad)
        ax.set_xlabel(titulo)
        ax.set_xticks(range(0, 24, 2))
        ax.grid(alpha=0.25)
        ax.legend()
    fig.suptitle(f"Error meteorológico y cobertura — {titulo}")
    fig.tight_layout()
    return fig

fig_utc_03 = graficar_horas(resumen_utc_03, "hora_utc", "hora objetivo UTC")
plt.show()
fig_solar_03 = graficar_horas(resumen_solar_03, "hora_solar", "hora solar media local aprox.")
plt.show()

## 5. Comparaciones de horarios con estaciones comunes

Se contrastan los pares solares **07 vs 14** y **15 vs 17**. Se reportan dos estimandos distintos:

1. **Estaciones comunes:** mínimo cinco observaciones por estación y hora; cada estación pesa lo mismo. Controla parcialmente la composición geográfica, pero puede comparar días diferentes.
2. **Estación y fecha comunes:** se emparejan también los días objetivo; cada par estación-día aporta una comparación. Reduce la confusión por fecha, pero **no garantiza igualdad de condiciones meteorológicas ni independencia de las unidades**.

Las diferencias son **descriptivas y retrospectivas**, no experimentos causales. Un resultado negativo implica que la segunda hora presenta un MAE menor que la primera.

In [ ]:
# 03.7 — Comparación con estaciones comunes, y luego estación × fecha común

def comparar_horas_comunes(datos: pd.DataFrame, hora_a: int, hora_b: int,
                          min_por_estacion: int = 5) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Contrasta MAE/sesgo entre horas sin confundir promedios por estación con pares fecha."""
    base = datos.loc[datos["hora_solar"].isin([hora_a, hora_b])].copy()
    g = (base.groupby(["horizon_target_hours", "station_id", "hora_solar"], observed=True)
         .agg(n=("error_c", "size"), mae=("error_abs_c", "mean"),
              sesgo=("error_c", "mean")).reset_index())
    a = g.loc[g.hora_solar.eq(hora_a)].drop(columns="hora_solar").rename(
        columns={"n": "n_a", "mae": "mae_a", "sesgo": "sesgo_a"})
    b = g.loc[g.hora_solar.eq(hora_b)].drop(columns="hora_solar").rename(
        columns={"n": "n_b", "mae": "mae_b", "sesgo": "sesgo_b"})
    pares = a.merge(b, on=["horizon_target_hours", "station_id"], validate="one_to_one")
    pares = pares.loc[(pares.n_a >= min_por_estacion) & (pares.n_b >= min_por_estacion)]

    # Una pareja estación-fecha puede tener más de un registro por hora: agregar por celda.
    gd = (base.groupby(["horizon_target_hours", "station_id", "fecha_utc", "hora_solar"],
                       observed=True).agg(mae=("error_abs_c", "mean"),
                                          sesgo=("error_c", "mean")).reset_index())
    da = gd.loc[gd.hora_solar.eq(hora_a)].drop(columns="hora_solar").rename(
        columns={"mae": "mae_a", "sesgo": "sesgo_a"})
    db = gd.loc[gd.hora_solar.eq(hora_b)].drop(columns="hora_solar").rename(
        columns={"mae": "mae_b", "sesgo": "sesgo_b"})
    pares_dia = da.merge(db, on=["horizon_target_hours", "station_id", "fecha_utc"],
                         validate="one_to_one")
    salida = []
    salida_dia = []
    for h in [12, 24]:
        p = pares.loc[pares.horizon_target_hours.eq(h)]
        q = pares_dia.loc[pares_dia.horizon_target_hours.eq(h)]
        salida.append({
            "comparacion": f"{hora_a:02d} vs {hora_b:02d}", "horizonte_h": h,
            "estaciones_comunes": p.station_id.nunique(),
            "mae_hora_a_c": p.mae_a.mean(), "mae_hora_b_c": p.mae_b.mean(),
            "delta_mae_b_menos_a_c": (p.mae_b - p.mae_a).mean(),
            "sesgo_hora_a_c": p.sesgo_a.mean(), "sesgo_hora_b_c": p.sesgo_b.mean(),
            "min_observaciones_por_estacion_hora": min_por_estacion,
        })
        salida_dia.append({
            "comparacion": f"{hora_a:02d} vs {hora_b:02d}", "horizonte_h": h,
            "pares_estacion_fecha": len(q), "estaciones": q.station_id.nunique(),
            "dias": q.fecha_utc.nunique(), "mae_hora_a_c": q.mae_a.mean(),
            "mae_hora_b_c": q.mae_b.mean(),
            "delta_mae_b_menos_a_c": (q.mae_b - q.mae_a).mean(),
            "sesgo_hora_a_c": q.sesgo_a.mean(), "sesgo_hora_b_c": q.sesgo_b.mean(),
        })
    return pd.DataFrame(salida), pd.DataFrame(salida_dia)

comparaciones_macro = []
comparaciones_dia = []
for ha, hb in [(7, 14), (15, 17)]:
    m, d = comparar_horas_comunes(df_temporal, ha, hb)
    comparaciones_macro.append(m)
    comparaciones_dia.append(d)
df_emparejados_03 = pd.concat(comparaciones_macro, ignore_index=True)
df_estacion_fecha_03 = pd.concat(comparaciones_dia, ignore_index=True)
print("MISMAS ESTACIONES (PESOS IGUALES):")
display(df_emparejados_03.round(3))
print("MISMAS ESTACIONES Y FECHAS (PARES OBSERVADOS):")
display(df_estacion_fecha_03.round(3))

## 6. Evolución diaria y días excluidos

El error puede variar sustancialmente de un día a otro. La figura utilizará un calendario completo desde el 10/09 hasta el 01/11 con **valores faltantes reales** del 22–26/10: las líneas quedan interrumpidas, sin interpolación. Los días 19–21/10 pueden investigarse por error elevado, pero no se descartan por ese motivo; la exclusión 22–26/10 corresponde a calidad de datos y recomendación docente.

In [ ]:
# 03.8 — Resumen por día objetivo y horizonte (no incluye noviembre)
df_diario_03 = (
    df_temporal.groupby(["fecha_utc", "horizon_target_hours"])
    .agg(registros=("error_c", "size"), estaciones=("station_id", "nunique"),
         sesgo_medio_c=("error_c", "mean"), mae_c=("error_abs_c", "mean"))
    .reset_index().rename(columns={"horizon_target_hours": "horizonte_h"})
    .sort_values(["fecha_utc", "horizonte_h"])
)
assert df_diario_03["registros"].sum() == len(df_temporal)
assert df_diario_03["fecha_utc"].nunique() == 48
assert (df_diario_03.groupby("horizonte_h")["fecha_utc"].nunique() == 48).all()

calendario_03 = pd.date_range("2026-09-10", "2026-11-01", tz="UTC", freq="D")
excluidos_03 = {pd.Timestamp(x, tz="UTC") for x in EXCLUIDOS}
calendario_filas_03 = []
for dia in calendario_03:
    for h in [12, 24]:
        calendario_filas_03.append({
            "fecha_utc": dia, "horizonte_h": h,
            "estado_fecha": "excluido_calidad" if dia in excluidos_03 else "historico_valido",
        })
df_calendario_03 = pd.DataFrame(calendario_filas_03).merge(
    df_diario_03, on=["fecha_utc", "horizonte_h"], how="left", validate="one_to_one"
)
assert df_calendario_03.loc[df_calendario_03.estado_fecha.eq("excluido_calidad"),
                            "mae_c"].isna().all()
assert df_calendario_03.loc[df_calendario_03.estado_fecha.eq("historico_valido"),
                            "mae_c"].notna().all()
print("ESTADÍSTICAS DIARIAS:")
display(df_diario_03.groupby("horizonte_h").agg(
    mae_min_c=("mae_c", "min"), mae_mediana_c=("mae_c", "median"),
    mae_max_c=("mae_c", "max"), dias=("fecha_utc", "nunique")
).round(3))
print("DÍAS DE MAYOR MAE HISTÓRICO:")
display(df_diario_03.sort_values("mae_c", ascending=False).groupby("horizonte_h").head(3).round(3))

In [ ]:
# 03.9 — Figura diaria con discontinuidad explícita
fig_diario_03, ejes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for h in [12, 24]:
    g = df_calendario_03.loc[df_calendario_03.horizonte_h.eq(h)].sort_values("fecha_utc")
    ejes[0].plot(g.fecha_utc, g.mae_c, marker="o", markersize=3, label=f"{h} h")
    ejes[1].plot(g.fecha_utc, g.sesgo_medio_c, marker="o", markersize=3, label=f"{h} h")
ejes[1].axhline(0, linestyle="--", color="gray", linewidth=1)
for eje in ejes:
    eje.axvspan(pd.Timestamp("2026-10-22", tz="UTC"),
                 pd.Timestamp("2026-10-27", tz="UTC"), color="gray", alpha=0.13,
                 label="22–26/10 excluidos")
    eje.grid(alpha=0.25)
    eje.legend()
ejes[0].set_ylabel("MAE (°C)")
ejes[0].set_title("Variación diaria del error original")
ejes[1].set_ylabel("Sesgo (°C)")
ejes[1].set_xlabel("Fecha objetivo (UTC)")
ejes[1].xaxis.set_major_locator(mdates.DayLocator(interval=5))
ejes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m", tz=timezone.utc))
fig_diario_03.tight_layout()
plt.show()

## 7. Antecedentes históricos de error: estudio exploratorio, no features finales

La disponibilidad real no queda demostrada únicamente porque un error corresponda a una fecha anterior. Se requiere que el antecedente haya sido generado, observado y recogido antes de la **emisión de consulta**, y el Notebook 04 añade un **margen supuesto de dos horas** después de la hora de recolección.

Aquí estudiaremos los antecedentes cuyo **objetivo histórico** coincide exactamente con `snapshot_actual − 12 h` o `snapshot_actual − 24 h`, de la misma estación y horizonte. Se evalúa la misma consulta bajo cuatro escenarios descriptivos:

- Corte nominal `snapshot`, margen de observación **0 h** (comparable con la versión exploratoria anterior, sujeto a otras marcas).
- Corte nominal `snapshot`, margen **2 h**.
- Corte en **máximo de captura y generación del pronóstico actual**, margen **0 h**.
- Corte en emisión efectiva, margen **2 h**.

**La elección del antecedente no cambia entre escenarios:** solo cambia la regla de disponibilidad. Eso permite comparar el supuesto temporal sin introducir lags basados en marcas futuras. La latencia de 2 h es **un supuesto operativo, no un dato empíricamente verificado**. Las observaciones de los días excluidos no se incorporan como antecedentes.

**Advertencia crucial:** este lookup exacto 12/24 h **no es** el lag empleado por 04 (`lag_error_disponible_c`, edad máxima 72 h, indicador de faltante y consulta causal del último error conocido). No se exportan estas columnas como features del modelo; son diagnósticos de cobertura y sensibilidad temporal.

In [ ]:
# 03.10 — Lags exploratorios por objetivo fijo; consulta única y sensibilidad temporal
CLAVE_LAG_03 = ["station_id", "horizon_target_hours", "target_time_utc"]
assert df_temporal[CLAVE_LAG_03].notna().all().all()
assert not df_temporal.duplicated(CLAVE_LAG_03).any(), (
    "No es única la clave estación/horizonte/objetivo; revisar antes de buscar lags."
)

ANTECEDENTES_03 = df_temporal.set_index(CLAVE_LAG_03)[
    ["error_c", "forecast_generated_at_utc", "collection_time_utc",
     "observation_time_utc", "observation_collection_hour_utc"]
]


def construir_lags_sin_fuga(datos: pd.DataFrame, antecedentes: pd.DataFrame,
                            horas_lag: int, corte: str = "snapshot",
                            demora_observacion_h: float = 0.0) -> pd.DataFrame:
    """Consulta *exploratoria* por objetivo fijo. No sustituye el lag de Notebook 04.

    `corte` compara emisión nominal vs. efectiva; `demora_observacion_h` es un
    margen hipotético adicional tras la hora de recolección, no una latencia observada.
    """
    if corte not in ("snapshot", "emision_efectiva"):
        raise ValueError("corte debe ser 'snapshot' o 'emision_efectiva'")
    if demora_observacion_h < 0:
        raise ValueError("demora_observacion_h no puede ser negativa")
    consulta = pd.MultiIndex.from_arrays([
        datos["station_id"].to_numpy(), datos["horizon_target_hours"].to_numpy(),
        (datos["snapshot_time_utc"] - pd.Timedelta(hours=horas_lag)).to_numpy()
    ], names=CLAVE_LAG_03)
    cand = antecedentes.reindex(consulta).reset_index(drop=True)
    instante = (datos["snapshot_time_utc"] if corte == "snapshot"
                else datos["emision_efectiva_utc"]).reset_index(drop=True)
    presente = cand["error_c"].notna()
    disponible = (
        presente & instante.notna()
        & cand["forecast_generated_at_utc"].lt(instante)
        & cand["collection_time_utc"].lt(instante)
        & cand["observation_time_utc"].lt(instante)
        & cand["observation_collection_hour_utc"]
          .add(pd.Timedelta(hours=demora_observacion_h)).lt(instante)
    )
    estado = np.select(
        [~presente, presente & ~disponible],
        ["sin_antecedente", "antecedente_no_disponible"], default="disponible"
    )
    return pd.DataFrame({
        "fecha_utc": datos["fecha_utc"].to_numpy(),
        "horizonte_h": datos["horizon_target_hours"].to_numpy(),
        "lag_horas": horas_lag, "corte": corte,
        "demora_observacion_h": demora_observacion_h,
        "estado": estado,
    })

filas_lags = []
for horas_lag in [12, 24]:
    for corte in ["snapshot", "emision_efectiva"]:
        for demora_h in [0.0, 2.0]:
            filas_lags.append(construir_lags_sin_fuga(
                df_temporal, ANTECEDENTES_03, horas_lag, corte, demora_h
            ))
df_lags_estados_03 = pd.concat(filas_lags, ignore_index=True)
del filas_lags

df_lags_resumen_03 = (
    df_lags_estados_03.groupby(
        ["horizonte_h", "lag_horas", "corte", "demora_observacion_h", "estado"]
    ).size().rename("registros").reset_index()
)
# Reestructurar estado en columnas para controles y barras.
df_lags_resumen_03 = df_lags_resumen_03.pivot_table(
    index=["horizonte_h", "lag_horas", "corte", "demora_observacion_h"],
    columns="estado", values="registros", aggfunc="sum", fill_value=0
).reset_index()
for columna in ["sin_antecedente", "antecedente_no_disponible", "disponible"]:
    if columna not in df_lags_resumen_03:
        df_lags_resumen_03[columna] = 0

df_lags_resumen_03["registros"] = df_lags_resumen_03[
    ["sin_antecedente", "antecedente_no_disponible", "disponible"]
].sum(axis=1)
df_lags_resumen_03["cobertura_pct"] = (
    100 * df_lags_resumen_03["disponible"] / df_lags_resumen_03["registros"]
)
for h in [12, 24]:
    assert (df_lags_resumen_03.loc[
        df_lags_resumen_03.horizonte_h.eq(h), "registros"
    ] == ESPERADO_HORIZ_03[h]).all()
print("SENSIBILIDAD DE DISPONIBILIDAD DE LAGS EXPLORATORIOS:")
display(df_lags_resumen_03.sort_values(
    ["horizonte_h", "lag_horas", "corte", "demora_observacion_h"]
).round(2))

In [ ]:
# 03.11 — Distinguir días sin antecedentes y discontinuidad 22–26/10
_df_lags_dias = (
    df_lags_estados_03.groupby(
        ["fecha_utc", "horizonte_h", "lag_horas", "corte",
         "demora_observacion_h", "estado"]
    ).size().rename("registros").reset_index()
)
df_lags_diarios_03 = _df_lags_dias.pivot_table(
    index=["fecha_utc", "horizonte_h", "lag_horas", "corte", "demora_observacion_h"],
    columns="estado", values="registros", aggfunc="sum", fill_value=0
).reset_index()
for col in ["sin_antecedente", "antecedente_no_disponible", "disponible"]:
    if col not in df_lags_diarios_03:
        df_lags_diarios_03[col] = 0

df_lags_diarios_03["total"] = df_lags_diarios_03[
    ["sin_antecedente", "antecedente_no_disponible", "disponible"]
].sum(axis=1)
df_lags_diarios_03["cobertura_pct"] = (
    100 * df_lags_diarios_03["disponible"] / df_lags_diarios_03["total"]
)
print("LAGS EXPLORATORIOS: CORTE NOMINAL / MARGEN 2 h, DÍAS SELECCIONADOS")
elegidos = pd.to_datetime([
    "2026-09-10", "2026-09-11", "2026-10-21", "2026-10-27", "2026-10-28"
], utc=True)
display(df_lags_diarios_03.loc[
    df_lags_diarios_03.fecha_utc.isin(elegidos)
    & df_lags_diarios_03.corte.eq("snapshot")
    & df_lags_diarios_03.demora_observacion_h.eq(2.0)
].round(2))

fig_lags_03, ax = plt.subplots(figsize=(10, 5))
bar_data = df_lags_resumen_03.loc[
    df_lags_resumen_03.corte.eq("snapshot")
    & df_lags_resumen_03.demora_observacion_h.eq(2.0)
].sort_values(["horizonte_h", "lag_horas"])
ax.bar(range(len(bar_data)), bar_data["cobertura_pct"])
ax.set_xticks(range(len(bar_data)))
ax.set_xticklabels([
    f"Pron. {int(h)} h / lag {int(l)} h"
    for h, l in zip(bar_data.horizonte_h, bar_data.lag_horas)
], rotation=15, ha="right")
ax.set_ylabel("Antecedentes disponibles (%)")
ax.set_title("Lags de objetivo fijo — exploración con corte nominal y margen supuesto 2 h")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
fig_lags_03.tight_layout()
plt.show()

## 8. Exportaciones para el dashboard y controles de coherencia

Los **CSV** contienen los números que utilizará el dashboard para presentar evolución diaria, cobertura solar, comparación de estaciones y cautelas de disponibilidad. Los **PNG** son únicamente referencias visuales; el dashboard podrá regenerarlos interactivamente a partir de los CSV. El **Parquet consolidado de 00** sigue siendo el detalle histórico central; el 03 no crea una segunda copia ni escribe sobre el test.

Los lags exploratorios se exportan **solo como agregados de cobertura**, no como features predictoras de 04. La huella de la cohorte permite detectar mezclas entre versiones. No se hacen inferencias de causalidad ni de estacionalidad anual con estos resultados.

In [ ]:
# 03.12 — Exportación estable, cotejo contra 00 y guía de visualizaciones
DIR_03.mkdir(parents=True, exist_ok=True)

tablas_03 = {
    "01_auditoria_relojes.csv": df_auditoria_relojes_03,
    "02_resumen_hora_utc.csv": resumen_utc_03,
    "03_resumen_hora_solar.csv": resumen_solar_03,
    "04_comparacion_estaciones_comunes.csv": df_emparejados_03,
    "05_comparacion_estacion_fecha.csv": df_estacion_fecha_03,
    "06_resumen_error_diario.csv": df_diario_03,
    "07_calendario_historico_con_exclusiones.csv": df_calendario_03,
    "08_lags_exploratorios_escenarios.csv": df_lags_resumen_03,
    "09_lags_exploratorios_por_dia.csv": df_lags_diarios_03,
}
for nombre, tabla in tablas_03.items():
    tabla.to_csv(DIR_03 / nombre, index=False, encoding="utf-8-sig")

imagenes_03 = {
    "10_error_hora_utc.png": fig_utc_03,
    "11_error_hora_solar.png": fig_solar_03,
    "12_error_diario_con_hueco.png": fig_diario_03,
    "13_disponibilidad_lags_exploratorios.png": fig_lags_03,
}
for nombre, figura in imagenes_03.items():
    figura.savefig(DIR_03 / nombre, dpi=160, bbox_inches="tight")
    plt.close(figura)

meta_03 = {
    "run_id": RUN_ID_03,
    "notebook": "03_analisis_temporal_solar.ipynb",
    "fuente": fuente_03,
    "historico_claves_sha256": HUELLA_03,
    "n_archivos_inventario": 61,
    "n_archivos_historicos": 48,
    "n_archivos_excluidos": 5,
    "n_archivos_test_reservado": 8,
    "filas_historicas": len(df_temporal),
    "filas_por_horizonte": {str(k): int(v) for k, v in
                            df_temporal.horizon_target_hours.value_counts().sort_index().items()},
    "regla_muestra": "match_type=interval; horizontes 12/24 h; temperaturas finitas",
    "dias_excluidos": "2026-10-22 al 2026-10-26 inclusive",
    "lag_03": "Exploración por objetivo exacto 12/24 h anterior a snapshot; no es el lag del modelo 04",
    "lag_04": "Último error pasado considerado disponible, edad <=72 h y latencia supuesta 2 h",
    "latencia": "La disponibilidad de observaciones real no está acreditada por los CSV",
    "hora_solar": "UTC del inicio del intervalo objetivo + longitud/15 (mod 24)",
    "test_no_consultado": True,
}
(DIR_03 / "14_manifiesto_03.json").write_text(
    json.dumps(meta_03, ensure_ascii=False, indent=2), encoding="utf-8"
)
assert all((DIR_03 / n).is_file() for n in (*tablas_03, *imagenes_03, "14_manifiesto_03.json"))

print("CARPETA DE EXPORTACIÓN:", DIR_03)
print("Tablas:", len(tablas_03), "| Gráficos:", len(imagenes_03), "| JSON: 1")
print("Filas históricas:", len(df_temporal))
print("Cohorte verificada con Notebook 00:", KPI_00.is_file())
print("No se leyeron registros del test de noviembre ni se entrenaron modelos.")

## 9. Conclusiones y vínculo con los otros notebooks

**Qué permite evaluar este notebook tras ejecutarlo con los 48 días históricos:**

- Las tablas de **hora UTC y solar** indican asociaciones entre el momento objetivo y el error original, y muestran cuántas estaciones y días sustentan cada hora. Una hora con baja cobertura exige cautela.
- Las comparaciones **dentro de estaciones** y **dentro de estación-fecha** reducen parcialmente problemas de composición; no demuestran causalidad meteorológica.
- La evolución diaria documenta variaciones de MAE y sesgo y distingue los cinco días excluidos. Un sesgo global cercano a cero puede ocultar sesgos diarios de distinto signo.
- Las variables horarias seno/coseno representan una periodicidad matemática conocida, pero no bastan para afirmar que su inclusión mejorará el modelo. Esa evaluación corresponde al **Notebook 04** y su validación temporal.
- Los lags fijos 12/24 h se conservan únicamente como **estudio conceptual**. Sus faltantes pueden provenir del comienzo de la serie, del hueco 22–26/10, de la estructura de cobertura o de condiciones temporales. El dato disponible más reciente utilizado por el modelo 04 tiene **otra definición**.

**No demostrado aquí:** latencia real de observaciones, predicción operativa en vivo, independencia prospectiva plena del test, causalidad física de ciclos horarios o comportamiento del modelo en otras estaciones/años.

**Dashboard:** emplear `02_resumen_hora_utc.csv`, `03_resumen_hora_solar.csv`, `05_comparacion_estacion_fecha.csv`, `07_calendario_historico_con_exclusiones.csv` y `08_lags_exploratorios_escenarios.csv` para explicar la ingeniería temporal, la calidad de la muestra y sus límites. No introducir estos agregados como si fueran resultados del modelo corregido ni convertir porcentajes de lag en confianza predictiva.

**Orden:** Notebook 00 (cohorte) → 01 (estaciones) → 02 (relaciones) → **03 (temporal)** → 04 (modelado). Si cambia la cohorte 00, volver a ejecutar 01–03 y revisar trazabilidad antes de continuar.